# Phase 4: conditional association rules

Run 4A and 4B in order with the official CSV and the matching phase 3 result ZIP. Item descriptions below are English glosses, not official questionnaire wording.


In [ ]:
import os, io, json, zipfile, hashlib, platform, sys, warnings, re
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import fisher_exact

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

INPUT_CSV_PATH = os.environ.get('PHASE4_INPUT_CSV', '/content/drive/MyDrive/SOICT-2026/data/soict_2026_survey_responses.csv')
PHASE3_RESULTS_PATH = os.environ.get('PHASE4_PHASE3_RESULTS', '')
if not PHASE3_RESULTS_PATH:
    raise ValueError('A matching phase 3 ZIP must be supplied in PHASE4_PHASE3_RESULTS.')
OUTPUT_BASE_DIR = os.environ.get('PHASE4_OUTPUT_DIR', '/content/drive/MyDrive/SOICT-2026/results/phase_04')
MIN_SUPPORT = float(os.environ.get('PHASE4_MIN_SUPPORT', '0.05'))
MIN_CONFIDENCE = 0.70
MIN_LIFT = 1.10
FDR_ALPHA = 0.05
MAX_ANTECEDENT_SIZE = 2
PAIR_REQUIRE_DIFFERENT_SCALES = True
MIN_CONFIDENCE_GAIN_PAIR = 0.05
TOP_SINGLE_PER_CONSEQUENT = 3
TOP_PAIR_PER_CONSEQUENT = 5
N_BOOTSTRAP = int(os.environ.get('PHASE4_BOOTSTRAP', '1000'))
RANDOM_SEED = 20260913
BOOTSTRAP_REQUIRED_LIFT_GT_1_RATE = 0.95
BOOTSTRAP_REQUIRED_FULL_THRESHOLD_RATE = 0.80
SENSITIVITY_MIN_COUNT = 10
SENSITIVITY_MIN_CONFIDENCE = 0.65
SENSITIVITY_MIN_LIFT = 1.0
PROFILE_MIN_ANTECEDENT_COUNT = 15
PROFILE_MIN_CONFIDENCE_DIFFERENCE = 0.15
FIGURE_DPI = 180
if not (0 < MIN_SUPPORT < 1) or N_BOOTSTRAP < 100:
    raise ValueError('MIN_SUPPORT must be between 0 and 1; N_BOOTSTRAP must be at least 100.')

input_path=Path(INPUT_CSV_PATH).expanduser()
if not input_path.is_file() or input_path.suffix.lower()!='.csv':
    raise FileNotFoundError(f'Input CSV not found: {input_path}')
output_root=Path(OUTPUT_BASE_DIR).expanduser()
output_root.mkdir(parents=True,exist_ok=True)
RUN_STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
phase4_root=output_root/f'phase_04_{RUN_STAMP}'
phase4_root.mkdir(exist_ok=False)
DIR_4A=phase4_root/'4A'
DIR_4B=phase4_root/'4B'
DIR_4A.mkdir(); DIR_4B.mkdir()
RUN_DIR=DIR_4B
INPUT_DIR=DIR_4A
CLUSTER_PATH=None
sns.set_theme(style='whitegrid')
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value.to_string(index=False) if hasattr(value, 'to_string') else value)

def sha256_file(path):
    digest=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):
            digest.update(part)
    return digest.hexdigest()

def save_csv(data,name,folder=DIR_4A):
    data.to_csv(folder/name,index=False,encoding='utf-8-sig')

def read_csv_robust(path,required=None):
    required=set(required or [])
    for enc in ('utf-8-sig','utf-8','cp1258','cp1252','latin1'):
        try:
            with open(path,encoding=enc) as f: header=f.readline()
            for sep in sorted((',', ';', '\t'),key=header.count,reverse=True):
                frame=pd.read_csv(path,encoding=enc,sep=sep)
                frame.columns=[str(c).replace('\ufeff','').strip() for c in frame.columns]
                if required.issubset(frame.columns) and len(frame.columns)>1:
                    return frame,{'encoding':enc,'separator':sep,'n_columns':len(frame.columns)}
        except (UnicodeError,ValueError,pd.errors.ParserError):
            continue
    raise ValueError(f'CSV is unreadable or required columns are missing: {path}')

print('CSV:',input_path,'Output:',phase4_root)


## 4A. Define item states and validate the CSV


In [ ]:
CODEBOOK_CSV = 'ITEM,TIER,SCALE,ANCHOR_TYPE,ROLE,SOURCE_QUESTION,PRIMARY_LOW_VALUES,PRIMARY_HIGH_VALUES,STRICT_LOW_VALUES,STRICT_HIGH_VALUES,LOW_TOKEN,HIGH_TOKEN,NEUTRAL_VALUE_3,MISSING_HANDLING,ANTECEDENT_ELIGIBLE,CONSEQUENT_ELIGIBLE\r\nENP01,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP01. I need English to communicate with other people.],1;2,4;5,1,5,ENP01__NHU_CAU_THAP,ENP01__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nENP02,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP02. I need English to access up-to-date information.],1;2,4;5,1,5,ENP02__NHU_CAU_THAP,ENP02__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nENP03,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP03. I need English for my future career.],1;2,4;5,1,5,ENP03__NHU_CAU_THAP,ENP03__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nENP04,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP04. I need English to access scholarship opportunities.],1;2,4;5,1,5,ENP04__NHU_CAU_THAP,ENP04__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nENP05,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP05. I need English to complete coursework in my field of study.],1;2,4;5,1,5,ENP05__NHU_CAU_THAP,ENP05__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nENP06,I,ENP,MUC_CAN_THIET,ANTECEDENT,[ENP06. I need English to meet the language requirements of my academic program.],1;2,4;5,1,5,ENP06__NHU_CAU_THAP,ENP06__NHU_CAU_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED01,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED01. I listen to and understand lectures in my field that are delivered in English.],1;2,4;5,1,5,PTED01__KHO_KHAN_THAP,PTED01__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED02,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED02. I read and understand materials in my field that are written in English.],1;2,4;5,1,5,PTED02__KHO_KHAN_THAP,PTED02__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED03,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED03. I use English to answer questions in written examinations for courses in my field.],1;2,4;5,1,5,PTED03__KHO_KHAN_THAP,PTED03__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED04,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED04. I use English to discuss topics in my field of study.],1;2,4;5,1,5,PTED04__KHO_KHAN_THAP,PTED04__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED05,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED05. I use English to answer questions in oral examinations.],1;2,4;5,1,5,PTED05__KHO_KHAN_THAP,PTED05__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nPTED06,I,PTED,MUC_DO_KHO_KHAN,ANTECEDENT,[PTED06. I use English to complete writing assignments for courses in my field.],1;2,4;5,1,5,PTED06__KHO_KHAN_THAP,PTED06__KHO_KHAN_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMEU01,I,MEU,TAN_SUAT,ANTECEDENT,[MEU01. I read materials in my field of study in English.],1;2,4;5,1,5,MEU01__TAN_SUAT_THAP,MEU01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMEU02,I,MEU,TAN_SUAT,ANTECEDENT,[MEU02. I listen to learning content in my field of study in English.],1;2,4;5,1,5,MEU02__TAN_SUAT_THAP,MEU02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMEU03,I,MEU,TAN_SUAT,ANTECEDENT,[MEU03. I use English to complete written assignments for courses in my field.],1;2,4;5,1,5,MEU03__TAN_SUAT_THAP,MEU03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMEU04,I,MEU,TAN_SUAT,ANTECEDENT,[MEU04. I summarize information from English materials for coursework in my field.],1;2,4;5,1,5,MEU04__TAN_SUAT_THAP,MEU04__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMEU05,I,MEU,TAN_SUAT,ANTECEDENT,[MEU05. I use English to discuss topics in my field of study.],1;2,4;5,1,5,MEU05__TAN_SUAT_THAP,MEU05__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nREH01,II,REH,TAN_SUAT,ANTECEDENT,[REH01. I repeat the material I am learning aloud to remember it.],1;2,4;5,1,5,REH01__TAN_SUAT_THAP,REH01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nREH02,II,REH,TAN_SUAT,ANTECEDENT,[REH02. I reread learning materials several times.],1;2,4;5,1,5,REH02__TAN_SUAT_THAP,REH02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nREH03,II,REH,TAN_SUAT,ANTECEDENT,[REH03. I remember keywords in English materials to recall important ideas.],1;2,4;5,1,5,REH03__TAN_SUAT_THAP,REH03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nELA01,II,ELA,TAN_SUAT,ANTECEDENT,[ELA01. I connect new material with what I already know.],1;2,4;5,1,5,ELA01__TAN_SUAT_THAP,ELA01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nELA02,II,ELA,TAN_SUAT,ANTECEDENT,[ELA02. I summarize the main ideas of the English materials I have studied.],1;2,4;5,1,5,ELA02__TAN_SUAT_THAP,ELA02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nELA03,II,ELA,TAN_SUAT,ANTECEDENT,[ELA03. I combine information from several learning sources.],1;2,4;5,1,5,ELA03__TAN_SUAT_THAP,ELA03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nORG01,II,ORG,TAN_SUAT,ANTECEDENT,[ORG01. I identify important ideas in the material I am studying.],1;2,4;5,1,5,ORG01__TAN_SUAT_THAP,ORG01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nORG02,II,ORG,TAN_SUAT,ANTECEDENT,[ORG02. I make an outline to organize the material I am studying.],1;2,4;5,1,5,ORG02__TAN_SUAT_THAP,ORG02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nORG03,II,ORG,TAN_SUAT,ANTECEDENT,[ORG03. I use tables or diagrams to organize the material I am studying.],1;2,4;5,1,5,ORG03__TAN_SUAT_THAP,ORG03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nCRT01,II,CRT,TAN_SUAT,ANTECEDENT,[CRT01. I evaluate how convincing the English-language information I encounter is.],1;2,4;5,1,5,CRT01__TAN_SUAT_THAP,CRT01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nCRT02,II,CRT,TAN_SUAT,ANTECEDENT,[CRT02. I examine evidence supporting an explanation or conclusion.],1;2,4;5,1,5,CRT02__TAN_SUAT_THAP,CRT02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nCRT03,II,CRT,TAN_SUAT,ANTECEDENT,[CRT03. I consider alternative explanations for a claim or conclusion.],1;2,4;5,1,5,CRT03__TAN_SUAT_THAP,CRT03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMSR01,II,MSR,TAN_SUAT,ANTECEDENT,[MSR01. I set specific goals before each independent English study session.],1;2,4;5,1,5,MSR01__TAN_SUAT_THAP,MSR01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMSR02,II,MSR,TAN_SUAT,ANTECEDENT,[MSR02. I ask myself questions to check my understanding while studying.],1;2,4;5,1,5,MSR02__TAN_SUAT_THAP,MSR02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMSR03,II,MSR,TAN_SUAT,ANTECEDENT,[MSR03. I identify what I do not fully understand while studying English independently.],1;2,4;5,1,5,MSR03__TAN_SUAT_THAP,MSR03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nMSR04,II,MSR,TAN_SUAT,ANTECEDENT,[MSR04. I change my study method when the current method does not work.],1;2,4;5,1,5,MSR04__TAN_SUAT_THAP,MSR04__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nTSEM01,II,TSEM,TAN_SUAT,ANTECEDENT,[TSEM01. I reserve specific times during the week for independent English study.],1;2,4;5,1,5,TSEM01__TAN_SUAT_THAP,TSEM01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nTSEM02,II,TSEM,TAN_SUAT,ANTECEDENT,[TSEM02. I maintain the independent English study schedule I set.],1;2,4;5,1,5,TSEM02__TAN_SUAT_THAP,TSEM02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nTSEM03,II,TSEM,TAN_SUAT,ANTECEDENT,[TSEM03. I choose a place to study English that limits distractions.],1;2,4;5,1,5,TSEM03__TAN_SUAT_THAP,TSEM03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS01,II,VLS,TAN_SUAT,ANTECEDENT,[VLS01. I use new English words in sentences to remember their meanings and usage.],1;2,4;5,1,5,VLS01__TAN_SUAT_THAP,VLS01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS02,II,VLS,TAN_SUAT,ANTECEDENT,[VLS02. I say or write new English words repeatedly to remember them.],1;2,4;5,1,5,VLS02__TAN_SUAT_THAP,VLS02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS03,II,VLS,TAN_SUAT,ANTECEDENT,[VLS03. I use flashcards to remember new English words.],1;2,4;5,1,5,VLS03__TAN_SUAT_THAP,VLS03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS04,II,VLS,TAN_SUAT,ANTECEDENT,[VLS04. I practice pronouncing new English words to remember their pronunciation.],1;2,4;5,1,5,VLS04__TAN_SUAT_THAP,VLS04__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS05,II,VLS,TAN_SUAT,ANTECEDENT,[VLS05. I review English words I have learned at different times.],1;2,4;5,1,5,VLS05__TAN_SUAT_THAP,VLS05__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nVLS06,II,VLS,TAN_SUAT,ANTECEDENT,[VLS06. I try to infer the meaning of unfamiliar English words.],1;2,4;5,1,5,VLS06__TAN_SUAT_THAP,VLS06__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL01,II,DEL,TAN_SUAT,ANTECEDENT,[DEL01. I use digital tools or resources to learn more about English.],1;2,4;5,1,5,DEL01__TAN_SUAT_THAP,DEL01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL02,II,DEL,TAN_SUAT,ANTECEDENT,[DEL02. I use technology to sustain English study toward my goals.],1;2,4;5,1,5,DEL02__TAN_SUAT_THAP,DEL02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL03,II,DEL,TAN_SUAT,ANTECEDENT,[DEL03. I use technology to create opportunities to practice English outside class.],1;2,4;5,1,5,DEL03__TAN_SUAT_THAP,DEL03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL04,II,DEL,TAN_SUAT,ANTECEDENT,[DEL04. I actively search online for English materials to support my studies.],1;2,4;5,1,5,DEL04__TAN_SUAT_THAP,DEL04__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL05,II,DEL,TAN_SUAT,ANTECEDENT,[DEL05. I use learning applications to practice English topics I need to improve.],1;2,4;5,1,5,DEL05__TAN_SUAT_THAP,DEL05__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL06,II,DEL,TAN_SUAT,ANTECEDENT,[DEL06. I select digital tools that fit my independent English study goals.],1;2,4;5,1,5,DEL06__TAN_SUAT_THAP,DEL06__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nDEL07,II,DEL,TAN_SUAT,ANTECEDENT,[DEL07. I ask an AI tool to explain something I do not understand while studying English independently.],1;2,4;5,1,5,DEL07__TAN_SUAT_THAP,DEL07__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,1,0\r\nLPSN01,III,LPSN,MUC_CAN_THIET,CONSEQUENT,[LPSN01. I need the application to clarify the objective of each learning task.],1;2,4;5,1,5,LPSN01__NHU_CAU_CHUC_NANG_THAP,LPSN01__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nLPSN02,III,LPSN,MUC_CAN_THIET,CONSEQUENT,[LPSN02. I need the application to show my learning progress.],1;2,4;5,1,5,LPSN02__NHU_CAU_CHUC_NANG_THAP,LPSN02__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nLPSN03,III,LPSN,MUC_CAN_THIET,CONSEQUENT,[LPSN03. I need the application to show statistics on my learning results.],1;2,4;5,1,5,LPSN03__NHU_CAU_CHUC_NANG_THAP,LPSN03__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nLPSN04,III,LPSN,MUC_CAN_THIET,CONSEQUENT,[LPSN04. I need the application to order tasks from suitable for my current ability to more difficult.],1;2,4;5,1,5,LPSN04__NHU_CAU_CHUC_NANG_THAP,LPSN04__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nLPSN05,III,LPSN,MUC_CAN_THIET,CONSEQUENT,[LPSN05. I need the application to offer tasks that let me learn with others toward a shared goal.],1;2,4;5,1,5,LPSN05__NHU_CAU_CHUC_NANG_THAP,LPSN05__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nSFN01,III,SFN,MUC_CAN_THIET,CONSEQUENT,[SFN01. I need the application to suggest how to continue a task when I encounter difficulties.],1;2,4;5,1,5,SFN01__NHU_CAU_CHUC_NANG_THAP,SFN01__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nSFN02,III,SFN,MUC_CAN_THIET,CONSEQUENT,[SFN02. I need the application to provide examples when I do not understand how to perform a task.],1;2,4;5,1,5,SFN02__NHU_CAU_CHUC_NANG_THAP,SFN02__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nSFN03,III,SFN,MUC_CAN_THIET,CONSEQUENT,[SFN03. I need the application to indicate which parts of my response do not meet the requirements.],1;2,4;5,1,5,SFN03__NHU_CAU_CHUC_NANG_THAP,SFN03__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nSFN04,III,SFN,MUC_CAN_THIET,CONSEQUENT,[SFN04. I need the application to explain how to revise an inadequate response after I finish a task.],1;2,4;5,1,5,SFN04__NHU_CAU_CHUC_NANG_THAP,SFN04__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN01,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN01. I need the application to select learning content that matches my learning goals.],1;2,4;5,1,5,TECN01__NHU_CAU_CHUC_NANG_THAP,TECN01__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN02,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN02. I need the application to present content in a way that matches my understanding.],1;2,4;5,1,5,TECN02__NHU_CAU_CHUC_NANG_THAP,TECN02__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN03,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN03. I need the application to suggest the next task after I complete a learning activity.],1;2,4;5,1,5,TECN03__NHU_CAU_CHUC_NANG_THAP,TECN03__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN04,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN04. I need the application to explain why it makes a learning suggestion.],1;2,4;5,1,5,TECN04__NHU_CAU_CHUC_NANG_THAP,TECN04__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN05,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN05. I need control over which learning history information the application may use.],1;2,4;5,1,5,TECN05__NHU_CAU_CHUC_NANG_THAP,TECN05__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nTECN06,III,TECN,MUC_CAN_THIET,CONSEQUENT,[TECN06. I need the application to suggest review times based on my learning history.],1;2,4;5,1,5,TECN06__NHU_CAU_CHUC_NANG_THAP,TECN06__NHU_CAU_CHUC_NANG_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,1\r\nPU01,IV,PU,MUC_DONG_Y,VALIDATION,[PU01. I believe the application could help me complete independent English study tasks faster.],1;2,4;5,1,5,PU01__TAN_SUAT_THAP,PU01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nPU02,IV,PU,MUC_DONG_Y,VALIDATION,[PU02. I believe the application could improve the quality of my independent English study work.],1;2,4;5,1,5,PU02__TAN_SUAT_THAP,PU02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nPU03,IV,PU,MUC_DONG_Y,VALIDATION,[PU03. I believe the application could help me complete more study activities in the same time.],1;2,4;5,1,5,PU03__TAN_SUAT_THAP,PU03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nPU04,IV,PU,MUC_DONG_Y,VALIDATION,[PU04. I believe the application could make my independent English study more effective.],1;2,4;5,1,5,PU04__TAN_SUAT_THAP,PU04__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nBI01,IV,BI,MUC_DONG_Y,VALIDATION,[BI01. I intend to use the application for independent English study in the next few months.],1;2,4;5,1,5,BI01__TAN_SUAT_THAP,BI01__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nBI02,IV,BI,MUC_DONG_Y,VALIDATION,[BI02. I expect to use the application for independent English study when it becomes available in the next few months.],1;2,4;5,1,5,BI02__TAN_SUAT_THAP,BI02__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\nBI03,IV,BI,MUC_DONG_Y,VALIDATION,[BI03. I plan to use the application to support independent English study in the next few months.],1;2,4;5,1,5,BI03__TAN_SUAT_THAP,BI03__TAN_SUAT_CAO,KHONG_TAO_TOKEN,KHONG_TAO_TOKEN_KHONG_XEM_LA_0,0,0\r\n'

dictionary_df=pd.read_csv(io.StringIO(CODEBOOK_CSV))
if len(dictionary_df)!=71 or dictionary_df.ITEM.duplicated().any():
    raise AssertionError('The item dictionary must contain exactly 71 distinct item codes.')
# The codebook contains English item glosses; official wording is external.
required_items=dictionary_df.ITEM.tolist()
input_hash_before=sha256_file(input_path)
df,read_meta=read_csv_robust(input_path,required_items)
if df.columns.duplicated().any():raise ValueError('CSV column names must be unique.')
if 'RESPONDENT_ID' not in df:
    df.insert(0,'RESPONDENT_ID',[f'ROW_{i:08d}' for i in range(1,len(df)+1)])
if df.RESPONDENT_ID.isna().any() or df.RESPONDENT_ID.astype(str).duplicated().any():
    raise ValueError('Respondent IDs must be present and unique.')
df.RESPONDENT_ID=df.RESPONDENT_ID.astype(str)
if len(df)<30:raise ValueError('Association rule mining requires at least 30 respondents.')
errors=[]
for item in required_items:
    values=pd.to_numeric(df[item],errors='coerce')
    unexpected=df[item].notna() & values.isna()
    out_of_range=values.notna() & ~values.isin([1,2,3,4,5])
    if unexpected.any() or out_of_range.any():
        errors.append({'ITEM':item,'NONNUMERIC':int(unexpected.sum()),'OUTSIDE_1_TO_5':int(out_of_range.sum())})
    df[item]=values
save_csv(pd.DataFrame(errors,columns=['ITEM','NONNUMERIC','OUTSIDE_1_TO_5']),'02_danh_sach_loi_du_lieu.csv')
save_csv(pd.DataFrame(columns=['ITEM','INVALID_VALUE']),'02b_gia_tri_item_khong_hop_le.csv')
if errors:raise ValueError('Responses must use the 1 to 5 scale; inspect 02_danh_sach_loi_du_lieu.csv')
enp_items=dictionary_df.loc[dictionary_df.SCALE=='ENP','ITEM'].tolist()
enp_missing=df[enp_items].isna().all(axis=1)
if (df[enp_items].isna().any(axis=1)&~enp_missing).any():
    raise ValueError('Only the complete ENP block may be absent after survey routing.')
others=[x for x in required_items if x not in enp_items]
if df[others].isna().any().any():raise ValueError('An item outside ENP is missing.')
if sha256_file(input_path)!=input_hash_before:raise IOError('Input CSV changed during reading.')
main_rows=df.loc[~enp_missing].copy()
MAIN_N=len(main_rows)
TOTAL_N=len(df)
C_MAIN=f'MAIN_N{MAIN_N}_WITH_ENP'
C_NO_ENP=f'SENSITIVITY_N{TOTAL_N}_NO_ENP'
C_STRICT=f'STRICT_N{MAIN_N}_WITH_ENP'
selected=dictionary_df.loc[dictionary_df.ROLE.isin(['ANTECEDENT','CONSEQUENT'])].copy()
assert len(selected)==64 and selected.ANTECEDENT_ELIGIBLE.sum()==49 and selected.CONSEQUENT_ELIGIBLE.sum()==15
save_csv(dictionary_df,'05_tu_dien_ma_hoa_trang_thai.csv')
save_csv(dictionary_df[['ITEM','TIER','SCALE','ANCHOR_TYPE','ROLE','SOURCE_QUESTION']], '06_doi_chieu_item_noi_dung.csv')
checks=pd.DataFrame([
 {'CHECK':'SO_DONG','OBSERVED':TOTAL_N,'EXPECTED':'DYNAMIC','STATUS':'DAT'},
 {'CHECK':'SO_DONG_DU_ENP','OBSERVED':MAIN_N,'EXPECTED':'DYNAMIC','STATUS':'DAT'},
 {'CHECK':'SO_CAU_LIKERT','OBSERVED':len(required_items),'EXPECTED':71,'STATUS':'DAT'},
 {'CHECK':'THIEU_NGOAI_ENP','OBSERVED':int(df[others].isna().sum().sum()),'EXPECTED':0,'STATUS':'DAT'},
 {'CHECK':'LOI_GIA_TRI','OBSERVED':len(errors),'EXPECTED':0,'STATUS':'DAT'},
])
save_csv(checks,'01_kiem_tra_du_lieu_dau_vao.csv')
missing_stats=[]
for scale,group in dictionary_df.groupby('SCALE',sort=False):
    items=group.ITEM.tolist(); miss=df[items].isna()
    missing_stats.append({'SCALE':scale,'TIER':group.TIER.iloc[0],
       'N_PEOPLE_ANY_MISSING':int(miss.any(axis=1).sum()),
       'N_PEOPLE_ALL_MISSING':int(miss.all(axis=1).sum()),'N_MISSING_CELLS':int(miss.sum().sum())})
save_csv(pd.DataFrame(missing_stats),'03_tom_tat_du_lieu_thieu.csv')
(DIR_4A/'04_nhat_ky_kiem_tra.txt').write_text(f'CSV SHA-256: {input_hash_before}\nTotal: {TOTAL_N}; ENP complete: {MAIN_N}; ENP not applicable: {int(enp_missing.sum())}\n',encoding='utf8')
print('Total',TOTAL_N,'ENP complete',MAIN_N,'ENP not applicable',int(enp_missing.sum()))


In [ ]:
def make_matrix(source,items,low_values,high_values):
    columns={'RESPONDENT_ID':source.RESPONDENT_ID.to_numpy()}
    for row in items.itertuples():
        values=source[row.ITEM]
        columns[row.LOW_TOKEN]=values.isin(low_values).astype('uint8').to_numpy()
        columns[row.HIGH_TOKEN]=values.isin(high_values).astype('uint8').to_numpy()
    return pd.DataFrame(columns)

main_matrix=make_matrix(main_rows,selected,(1,2),(4,5))
no_enp_matrix=make_matrix(df,selected.loc[selected.SCALE!='ENP'],(1,2),(4,5))
strict_matrix=make_matrix(main_rows,selected,(1,),(5,))
assert main_matrix.shape==(MAIN_N,129) and no_enp_matrix.shape==(TOTAL_N,117)
assert strict_matrix.shape==main_matrix.shape
for frame in (main_matrix,no_enp_matrix,strict_matrix):
    for row in selected.itertuples():
        if row.LOW_TOKEN in frame and (frame[row.LOW_TOKEN]&frame[row.HIGH_TOKEN]).any():
            raise AssertionError('The two states of an item must be mutually exclusive.')
    if frame.RESPONDENT_ID.duplicated().any():raise AssertionError('Matrix respondent IDs must be unique.')
save_csv(main_matrix,'10a_ma_tran_nhi_phan_chinh.csv')
save_csv(no_enp_matrix,'10b_ma_tran_nhi_phan_khong_ENP.csv')
save_csv(strict_matrix,'10c_ma_tran_nhi_phan_nghiem_ngat.csv')
from scipy.sparse import csr_matrix,save_npz
save_npz(DIR_4A/'10d_ma_tran_thua_chinh.npz',csr_matrix(main_matrix.drop(columns='RESPONDENT_ID').to_numpy()))
save_csv(pd.DataFrame({'COLUMN_INDEX':range(main_matrix.shape[1]-1),'TOKEN':main_matrix.columns[1:]}),'10e_cot_ma_tran_thua_chinh.csv')

configs=[(C_MAIN,main_matrix,'07_giao_dich_chinh.csv'),
         (C_NO_ENP,no_enp_matrix,'08_giao_dich_khong_ENP.csv'),
         (C_STRICT,strict_matrix,'09_giao_dich_nguong_nghiem_ngat.csv')]
for label,frame,filename in configs:
    tokens=frame.columns[1:]
    matrix=frame[tokens].to_numpy(dtype=bool)
    transactions=[' | '.join(tokens[row]) for row in matrix]
    save_csv(pd.DataFrame({'RESPONDENT_ID':frame.RESPONDENT_ID,
       'CONFIGURATION':label,'N_ACTIVE_TOKENS':matrix.sum(axis=1),
       'TRANSACTION':transactions}),filename)
print('4A: three matrices and transaction sets are ready.')


## 4A. Inspect states and create the 4B reference


In [ ]:
frequency_rows=[]; baseline_rows=[]; sparsity_rows=[]
lookup=dictionary_df.set_index('ITEM')
for label,frame,_ in configs:
    n=len(frame);values=frame.drop(columns='RESPONDENT_ID').to_numpy(dtype=np.uint8)
    counts=values.sum(axis=0)
    sparsity_rows.append({'CONFIGURATION':label,'N_RESPONDENTS':n,'N_TOKEN_COLUMNS':len(counts),
      'PRESENT_CELLS':int(counts.sum()),'DENSITY':float(values.mean()),
      'ACTIVE_TOKENS_MEAN':float(values.sum(axis=1).mean()),
      'ACTIVE_TOKENS_MEDIAN':float(np.median(values.sum(axis=1))),
      'ACTIVE_TOKENS_MIN':int(values.sum(axis=1).min()),
      'ACTIVE_TOKENS_MAX':int(values.sum(axis=1).max())})
    for token,count in zip(frame.columns[1:],counts):
        item,state=token.split('__',1)
        r=lookup.loc[item];rate=float(count/n)
        row={'CONFIGURATION':label,'TOKEN':token,'ITEM':item,'SCALE':r.SCALE,
         'TIER':r.TIER,'ROLE':r.ROLE,'STATE':'LOW' if token==r.LOW_TOKEN else 'HIGH',
         'N':n,'COUNT':int(count),'RATE':rate,
         'RARE_LT_5PCT':int(rate<0.05),'COMMON_GT_90PCT':int(rate>0.9)}
        frequency_rows.append(row)
        if r.CONSEQUENT_ELIGIBLE==1 and token==r.HIGH_TOKEN:
            baseline_rows.append({'CONFIGURATION':label,'TOKEN':token,'ITEM':item,'SCALE':r.SCALE,
              'N':n,'COUNT':int(count),'BASELINE_RATE_HIGH_NEED':rate})
frequency_df=pd.DataFrame(frequency_rows)
baseline_df=pd.DataFrame(baseline_rows)
sparsity_df=pd.DataFrame(sparsity_rows)
save_csv(frequency_df,'11_tan_suat_tung_trang_thai.csv')
save_csv(baseline_df,'12_tan_suat_nhu_cau_chuc_nang.csv')
save_csv(frequency_df.loc[(frequency_df.RARE_LT_5PCT==1)|(frequency_df.COMMON_GT_90PCT==1)],'13_trang_thai_qua_hiem_hoac_qua_pho_bien.csv')
save_csv(sparsity_df,'14_tom_tat_do_thua_cau_hinh.csv')

antecedents=[token for token in main_matrix.columns[1:] if lookup.loc[token.split('__')[0],'ANTECEDENT_ELIGIBLE']==1]
consequents=[r.HIGH_TOKEN for r in selected.itertuples() if r.CONSEQUENT_ELIGIBLE==1]
n=len(main_matrix);arr=main_matrix
cooc=[]
for a in antecedents:
    x=arr[a].to_numpy(dtype=bool);nx=int(x.sum())
    for b in consequents:
        y=arr[b].to_numpy(dtype=bool);ny=int(y.sum());nxy=int((x&y).sum())
        conf=nxy/nx if nx else np.nan;base=ny/n
        cooc.append({'ANTECEDENT_TOKEN':a,'CONSEQUENT_TOKEN':b,'N':n,
          'ANTECEDENT_COUNT':nx,'CONSEQUENT_COUNT':ny,'JOINT_COUNT':nxy,
          'JOINT_RATE':nxy/n,'CONFIDENCE_PRELIMINARY':conf,
          'LIFT_PRELIMINARY':conf/base if base else np.nan,
          'STATUS':'CHAN_DOAN_KHONG_PHAI_LUAT_CUOI'})
save_csv(pd.DataFrame(cooc),'15_dong_xuat_hien_so_bo_T1_T2_voi_T3.csv')

fig,ax=plt.subplots(figsize=(9,4));ax.bar(sparsity_df.CONFIGURATION,sparsity_df.DENSITY,color='0.55');ax.tick_params(axis='x',rotation=15);ax.set_ylabel('Token density');fig.tight_layout();fig.savefig(DIR_4A/'19_so_sanh_do_thua_ba_cau_hinh.png',dpi=FIGURE_DPI);plt.close(fig)
fig,ax=plt.subplots(figsize=(11,4));ax.hist(frequency_df.loc[frequency_df.CONFIGURATION==C_MAIN,'RATE'],bins=20,color='0.5');ax.set_xlabel('State frequency');fig.tight_layout();fig.savefig(DIR_4A/'17_tan_suat_trang_thai_theo_thang.png',dpi=FIGURE_DPI);plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4));baseline_df.loc[baseline_df.CONFIGURATION==C_MAIN].plot.bar(x='ITEM',y='BASELINE_RATE_HIGH_NEED',ax=ax,color='0.55',legend=False);fig.tight_layout();fig.savefig(DIR_4A/'20_ty_le_nen_nhu_cau_chuc_nang.png',dpi=FIGURE_DPI);plt.close(fig)
fig,ax=plt.subplots(figsize=(10,4));ax.text(.05,.5,f'CSV → validate {TOTAL_N} rows → encode primary {MAIN_N} rows → two sensitivity checks',fontsize=12);ax.axis('off');fig.savefig(DIR_4A/'16_luong_chuan_bi_du_lieu.png',dpi=FIGURE_DPI);plt.close(fig)
fig,ax=plt.subplots(figsize=(11,4));top=pd.DataFrame(cooc).nlargest(15,'LIFT_PRELIMINARY');ax.barh(range(len(top)),top.LIFT_PRELIMINARY,color='0.5');ax.set_yticks(range(len(top)),[f'{r.ANTECEDENT_TOKEN.split("__")[0]} → {r.CONSEQUENT_TOKEN.split("__")[0]}' for r in top.itertuples()]);fig.tight_layout();fig.savefig(DIR_4A/'18_ban_do_dong_xuat_hien_so_bo.png',dpi=FIGURE_DPI);plt.close(fig)

config4a={'run_timestamp_utc':RUN_STAMP,'input_path':str(input_path),
 'input_sha256_before':input_hash_before,'input_sha256_after':sha256_file(input_path),
 'input_read_meta':read_meta,'n_rows':TOTAL_N,'n_complete_enp':MAIN_N,
 'codebook_source':'Historical 4A dictionary; six truncated TECN text entries replaced with official TECN.md supplied by user',
 'main_low_values':[1,2],'main_high_values':[4,5],
 'strict_low_values':[1],'strict_high_values':[5],
 'modeling_unit':'one CSV row','automatic_causal_conclusion':False}
(DIR_4A/'21_cau_hinh_lan_chay.json').write_text(json.dumps(config4a,ensure_ascii=False,indent=2),encoding='utf8')
(DIR_4A/'22_huong_dan_doc_ket_qua.md').write_text(f'# Phase 4A rule preparation\n\n{TOTAL_N} rows; {MAIN_N} complete on ENP. Primary encoding uses 1–2/4–5; strict sensitivity uses 1/5. Phase 4B tests co-occurrence without causal inference.\n',encoding='utf8')
readback=[]
for path in sorted(DIR_4A.iterdir()):
    if path.suffix=='.csv':
        try:pd.read_csv(path,encoding='utf-8-sig');ok=True
        except pd.errors.EmptyDataError:ok=path.name in ('02_danh_sach_loi_du_lieu.csv','02b_gia_tri_item_khong_hop_le.csv')
        readback.append({'FILE':path.name,'STATUS':'DAT' if ok else 'KHONG_DAT','DETAIL':''})
save_csv(pd.DataFrame(readback),'23_kiem_tra_doc_lai_dau_ra.csv')
if any(row['STATUS']!='DAT' for row in readback):raise AssertionError('A 4A output CSV cannot be read back.')
manifest=pd.DataFrame([{'FILE':p.name,'SIZE_BYTES':p.stat().st_size,'SHA256':sha256_file(p)} for p in sorted(DIR_4A.iterdir()) if p.is_file() and p.name!='24_manifest_sha256.csv'])
save_csv(manifest,'24_manifest_sha256.csv')
if sha256_file(input_path)!=input_hash_before:raise IOError('Input CSV changed during 4A.')
print('4A complete:',len(manifest),'files passed hash verification.')


## 4B. Mine and validate association rules


In [ ]:
FILES={'dictionary':'05_tu_dien_ma_hoa_trang_thai.csv','main':'10a_ma_tran_nhi_phan_chinh.csv',
 'no_enp':'10b_ma_tran_nhi_phan_khong_ENP.csv','strict':'10c_ma_tran_nhi_phan_nghiem_ngat.csv',
 'manifest':'24_manifest_sha256.csv'}
checks=pd.read_csv(INPUT_DIR/FILES['manifest'],encoding='utf-8-sig')
failures=[r.FILE for r in checks.itertuples() if not (INPUT_DIR/r.FILE).is_file() or sha256_file(INPUT_DIR/r.FILE)!=r.SHA256]
if failures:raise ValueError(f'4A output does not match its manifest: {failures}')
dictionary_df=pd.read_csv(INPUT_DIR/FILES['dictionary'],encoding='utf-8-sig')
main_matrix=pd.read_csv(INPUT_DIR/FILES['main'],encoding='utf-8-sig',dtype={'RESPONDENT_ID':str})
no_enp_matrix=pd.read_csv(INPUT_DIR/FILES['no_enp'],encoding='utf-8-sig',dtype={'RESPONDENT_ID':str})
strict_matrix=pd.read_csv(INPUT_DIR/FILES['strict'],encoding='utf-8-sig',dtype={'RESPONDENT_ID':str})
if len(main_matrix)!=MAIN_N or len(no_enp_matrix)!=TOTAL_N or len(strict_matrix)!=MAIN_N:
    raise ValueError('The 4A and 4B row counts do not match the current CSV.')
for m in (main_matrix,no_enp_matrix,strict_matrix):
    if m.RESPONDENT_ID.duplicated().any() or not m.drop(columns='RESPONDENT_ID').isin([0,1]).all().all():
        raise ValueError('The 4A matrix has invalid IDs or nonbinary values.')
if list(strict_matrix.columns)!=list(main_matrix.columns):raise ValueError('Strict matrix columns have the wrong order.')
if not set(no_enp_matrix.columns).issubset(main_matrix.columns):raise ValueError('The no ENP matrix has unexpected columns.')

cluster_labels=None
cluster_source={'provided':bool(PHASE3_RESULTS_PATH)}
if PHASE3_RESULTS_PATH:
    cluster_input=Path(PHASE3_RESULTS_PATH).expanduser()
    if not cluster_input.is_file():raise FileNotFoundError(cluster_input)
    if cluster_input.suffix.lower()=='.zip':
        with zipfile.ZipFile(cluster_input) as archive:
            matches=[x for x in archive.namelist() if x.startswith('3B/') and x.endswith('/02_silhouette_tung_nguoi.csv')]
            audits=[x for x in archive.namelist() if x.startswith('3A/') and x.endswith('/00_kiem_tra_dau_vao.json')]
            if len(matches)!=1 or len(audits)!=1:raise ValueError('The phase 3 ZIP must contain one 3B label file and the 3A input audit.')
            audit3=json.loads(archive.read(audits[0]))
            if audit3['sha256']!=input_hash_before:raise ValueError('The CSV SHA-256 differs from the phase 3 input.')
            cluster_labels=pd.read_csv(io.BytesIO(archive.read(matches[0])),encoding='utf-8-sig')
            cluster_source.update({'sha256_matched':True,'path':str(cluster_input)})
    elif cluster_input.suffix.lower()=='.csv':
        cluster_labels,meta=read_csv_robust(cluster_input,{'RESPONDENT_ID','PROFILE_CODE'})
        cluster_source.update({'sha256_matched':False,'path':str(cluster_input),'warning':'A standalone label CSV cannot establish the phase 3 input provenance.'})
    else:raise ValueError('PHASE3_RESULTS_PATH must point to a phase 3 ZIP or 3B label CSV.')
    if 'PROFILE_CODE' not in cluster_labels or cluster_labels.RESPONDENT_ID.duplicated().any() or set(cluster_labels.RESPONDENT_ID.astype(str))!=set(main_matrix.RESPONDENT_ID.astype(str)):
        raise ValueError('Phase 3 respondent IDs do not match the 4A primary sample.')
    cluster_labels.RESPONDENT_ID=cluster_labels.RESPONDENT_ID.astype(str)

pd.DataFrame([{'CHECK':'4A_MANIFEST','OBSERVED':len(failures),'EXPECTED':0,'STATUS':'DAT'},
 {'CHECK':'MAIN_N','OBSERVED':len(main_matrix),'EXPECTED':MAIN_N,'STATUS':'DAT'},
 {'CHECK':'NO_ENP_N','OBSERVED':len(no_enp_matrix),'EXPECTED':TOTAL_N,'STATUS':'DAT'}]).to_csv(RUN_DIR/'01_kiem_tra_dau_vao.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(columns=['FILE','PROBLEM']).to_csv(RUN_DIR/'02_loi_toan_ven_4A.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(columns=['PROBLEM']).to_csv(RUN_DIR/'03_loi_cau_truc.csv',index=False,encoding='utf-8-sig')
print('4B verified the 4A outputs and phase 3 labels:',cluster_source)


### Directional rules and multiple-testing correction


In [ ]:
dictionary_by_item = dictionary_df.set_index('ITEM').to_dict('index')
token_info = {}
for _, row in dictionary_df.iterrows():
    for state, column in [('LOW', 'LOW_TOKEN'), ('HIGH', 'HIGH_TOKEN')]:
        token_info[row[column]] = {
            'item': row['ITEM'],
            'scale': row['SCALE'],
            'tier': row['TIER'],
            'role': row['ROLE'],
            'state': state,
            'source_question': row.get('SOURCE_QUESTION', ''),
        }

antecedent_tokens = []
for _, row in dictionary_df.loc[dictionary_df['ANTECEDENT_ELIGIBLE'] == 1].iterrows():
    antecedent_tokens.extend([row['LOW_TOKEN'], row['HIGH_TOKEN']])
consequent_tokens = dictionary_df.loc[
    dictionary_df['CONSEQUENT_ELIGIBLE'] == 1, 'HIGH_TOKEN'
].tolist()

antecedent_tokens = [t for t in antecedent_tokens if t in main_matrix.columns]
consequent_tokens = [t for t in consequent_tokens if t in main_matrix.columns]

if len(antecedent_tokens) != 98:
    raise ValueError(f'Antecedent token count is {len(antecedent_tokens)}; expected 98.')
if len(consequent_tokens) != 15:
    raise ValueError(f'Consequent token count is {len(consequent_tokens)}; expected 15.')

def bh_adjust(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    n = len(pvalues)
    if n == 0:
        return np.array([], dtype=float)
    order = np.argsort(pvalues)
    ranked = pvalues[order] * n / np.arange(1, n + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted

def rule_metrics(x, y):
    x = np.asarray(x, dtype=bool)
    y = np.asarray(y, dtype=bool)
    n = len(x)
    n_x = int(x.sum())
    n_y = int(y.sum())
    n_xy = int(np.logical_and(x, y).sum())
    support_x = n_x / n
    support_y = n_y / n
    support = n_xy / n
    confidence = n_xy / n_x if n_x else np.nan
    lift = confidence / support_y if support_y else np.nan
    leverage = support - support_x * support_y
    conviction = (1 - support_y) / (1 - confidence) if confidence < 1 else np.inf
    table = [[n_xy, n_x - n_xy], [n_y - n_xy, n - n_x - n_y + n_xy]]
    p_value = fisher_exact(table, alternative='greater').pvalue
    return {
        'N': n,
        'ANTECEDENT_COUNT': n_x,
        'CONSEQUENT_COUNT': n_y,
        'JOINT_COUNT': n_xy,
        'ANTECEDENT_SUPPORT': support_x,
        'CONSEQUENT_BASELINE': support_y,
        'SUPPORT': support,
        'CONFIDENCE': confidence,
        'LIFT': lift,
        'LEVERAGE': leverage,
        'CONVICTION': conviction,
        'FISHER_P': p_value,
    }

def constrained_apriori(frame):
    n = len(frame)
    masks = {col: frame[col].to_numpy(dtype=bool) for col in antecedent_tokens + consequent_tokens}
    frequent_singletons = [token for token in antecedent_tokens if masks[token].mean() >= MIN_SUPPORT]
    candidate_sets = [(token,) for token in frequent_singletons]

    if MAX_ANTECEDENT_SIZE >= 2:
        for first, second in combinations(frequent_singletons, 2):
            if token_info[first]['item'] == token_info[second]['item']:
                continue
            if PAIR_REQUIRE_DIFFERENT_SCALES and token_info[first]['scale'] == token_info[second]['scale']:
                continue
            pair_mask = np.logical_and(masks[first], masks[second])
            if pair_mask.mean() >= MIN_SUPPORT:
                candidate_sets.append((first, second))

    rows = []
    for antecedents in candidate_sets:
        x = np.logical_and.reduce([masks[token] for token in antecedents])
        for consequent in consequent_tokens:
            metrics = rule_metrics(x, masks[consequent])
            if metrics['SUPPORT'] < MIN_SUPPORT:
                continue
            row = {
                'RULE_ID': '',
                'ANTECEDENT_SIZE': len(antecedents),
                'ANTECEDENT_1': antecedents[0],
                'ANTECEDENT_2': antecedents[1] if len(antecedents) == 2 else '',
                'ANTECEDENT_SET': ' & '.join(antecedents),
                'CONSEQUENT': consequent,
            }
            row.update(metrics)
            rows.append(row)

    rules = pd.DataFrame(rows)
    if rules.empty:
        rules = pd.DataFrame(columns=['RULE_ID','ANTECEDENT_SIZE','ANTECEDENT_1','ANTECEDENT_2','ANTECEDENT_SET','CONSEQUENT','N','ANTECEDENT_COUNT','CONSEQUENT_COUNT','JOINT_COUNT','ANTECEDENT_SUPPORT','CONSEQUENT_BASELINE','SUPPORT','CONFIDENCE','LIFT','LEVERAGE','CONVICTION','FISHER_P'])
    rules['FDR_Q'] = bh_adjust(rules['FISHER_P'].to_numpy())
    rules.sort_values(['LIFT', 'CONFIDENCE', 'SUPPORT'], ascending=False, inplace=True)
    rules.reset_index(drop=True, inplace=True)
    rules['RULE_ID'] = [f'R4B_{i:06d}' for i in range(1, len(rules) + 1)]
    return rules

candidate_rules = constrained_apriori(main_matrix)
candidate_rules.to_csv(RUN_DIR / '04_tat_ca_luat_ung_vien.csv', index=False, encoding='utf-8-sig')

print('Candidate rules meeting minimum support:', len(candidate_rules))
display(candidate_rules.head(10))


### Initial screening and antecedent pruning


In [ ]:
single_lookup = {
    (row.ANTECEDENT_1, row.CONSEQUENT): (row.CONFIDENCE, row.LIFT)
    for row in candidate_rules.itertuples()
    if row.ANTECEDENT_SIZE == 1
}

confidence_gain = []
lift_gain = []
for row in candidate_rules.itertuples():
    if row.ANTECEDENT_SIZE == 1:
        confidence_gain.append(np.nan)
        lift_gain.append(np.nan)
        continue
    subset_values = [
        single_lookup.get((token, row.CONSEQUENT), (np.nan, np.nan))
        for token in [row.ANTECEDENT_1, row.ANTECEDENT_2]
    ]
    best_conf = np.nanmax([value[0] for value in subset_values])
    best_lift = np.nanmax([value[1] for value in subset_values])
    confidence_gain.append(row.CONFIDENCE - best_conf)
    lift_gain.append(row.LIFT - best_lift)

candidate_rules['CONFIDENCE_GAIN_OVER_BEST_SUBRULE'] = confidence_gain
candidate_rules['LIFT_GAIN_OVER_BEST_SUBRULE'] = lift_gain
candidate_rules['PASS_EFFECT_THRESHOLDS'] = (
    (candidate_rules['CONFIDENCE'] >= MIN_CONFIDENCE)
    & (candidate_rules['LIFT'] >= MIN_LIFT)
    & (candidate_rules['LEVERAGE'] > 0)
)
candidate_rules['PASS_FDR'] = candidate_rules['FDR_Q'] <= FDR_ALPHA
candidate_rules['PASS_NONREDUNDANCY'] = (
    (candidate_rules['ANTECEDENT_SIZE'] == 1)
    | (candidate_rules['CONFIDENCE_GAIN_OVER_BEST_SUBRULE'] >= MIN_CONFIDENCE_GAIN_PAIR)
)
candidate_rules['PASS_INITIAL_SCREEN'] = (
    candidate_rules['PASS_EFFECT_THRESHOLDS']
    & candidate_rules['PASS_FDR']
    & candidate_rules['PASS_NONREDUNDANCY']
)

screened_rules = candidate_rules.loc[candidate_rules['PASS_INITIAL_SCREEN']].copy()
screened_rules.sort_values(['CONSEQUENT', 'ANTECEDENT_SIZE', 'LIFT', 'SUPPORT'], ascending=[True, True, False, False], inplace=True)

shortlist_parts = []
for consequent, block in screened_rules.groupby('CONSEQUENT'):
    singles = block.loc[block['ANTECEDENT_SIZE'] == 1].head(TOP_SINGLE_PER_CONSEQUENT)
    pairs = block.loc[block['ANTECEDENT_SIZE'] == 2].head(TOP_PAIR_PER_CONSEQUENT)
    shortlist_parts.extend([singles, pairs])
shortlisted_rules = pd.concat(shortlist_parts, ignore_index=True) if shortlist_parts else pd.DataFrame(columns=candidate_rules.columns)
shortlisted_rules.drop_duplicates('RULE_ID', inplace=True)
shortlisted_rules.sort_values(['LIFT', 'CONFIDENCE', 'SUPPORT'], ascending=False, inplace=True)

screen_summary = pd.DataFrame([
    {'STAGE': 'LUAT_UNG_VIEN_SUPPORT_DAT', 'N_RULES': len(candidate_rules)},
    {'STAGE': 'DAT_HIEU_QUA', 'N_RULES': int(candidate_rules['PASS_EFFECT_THRESHOLDS'].sum())},
    {'STAGE': 'DAT_HIEU_QUA_VA_FDR', 'N_RULES': int((candidate_rules['PASS_EFFECT_THRESHOLDS'] & candidate_rules['PASS_FDR']).sum())},
    {'STAGE': 'DAT_HIEU_QUA_FDR_VA_KHONG_DU_THUA', 'N_RULES': len(screened_rules)},
    {'STAGE': 'DUOC_CHON_KIEM_DINH_SAU', 'N_RULES': len(shortlisted_rules)},
])

candidate_rules.to_csv(RUN_DIR / '05_luat_ung_vien_kem_co_sang_loc.csv', index=False, encoding='utf-8-sig')
screened_rules.to_csv(RUN_DIR / '06_luat_dat_sang_loc_ban_dau.csv', index=False, encoding='utf-8-sig')
shortlisted_rules.to_csv(RUN_DIR / '07_luat_duoc_chon_kiem_dinh_sau.csv', index=False, encoding='utf-8-sig')
screen_summary.to_csv(RUN_DIR / '08_tom_tat_sang_loc.csv', index=False, encoding='utf-8-sig')

display(screen_summary)
print('Rules selected for bootstrap:', len(shortlisted_rules))

HAS_RULES = not shortlisted_rules.empty


### Respondent bootstrap


In [ ]:
if HAS_RULES:
    if shortlisted_rules.empty:
        raise RuntimeError('No rule passed the initial screening.')

    rng = np.random.default_rng(RANDOM_SEED)
    n_main = len(main_matrix)
    bootstrap_indices = rng.integers(0, n_main, size=(N_BOOTSTRAP, n_main))
    main_masks = {col: main_matrix[col].to_numpy(dtype=bool) for col in antecedent_tokens + consequent_tokens}

    bootstrap_rows = []
    for row in shortlisted_rules.itertuples():
        antecedents = [row.ANTECEDENT_1] + ([row.ANTECEDENT_2] if row.ANTECEDENT_SIZE == 2 else [])
        x = np.logical_and.reduce([main_masks[token] for token in antecedents])
        y = main_masks[row.CONSEQUENT]
        bx = x[bootstrap_indices]
        by = y[bootstrap_indices]
        count_x = bx.sum(axis=1)
        count_y = by.sum(axis=1)
        count_xy = np.logical_and(bx, by).sum(axis=1)
        support_b = count_xy / n_main
        with np.errstate(divide='ignore', invalid='ignore'):
            confidence_b = np.divide(count_xy, count_x, out=np.full(N_BOOTSTRAP, np.nan), where=count_x > 0)
            baseline_b = count_y / n_main
            lift_b = np.divide(confidence_b, baseline_b, out=np.full(N_BOOTSTRAP, np.nan), where=baseline_b > 0)
        valid = np.isfinite(lift_b) & np.isfinite(confidence_b)
        support_b = support_b[valid]
        confidence_b = confidence_b[valid]
        lift_b = lift_b[valid]
        bootstrap_rows.append({
            'RULE_ID': row.RULE_ID,
            'BOOTSTRAP_VALID_REPS': len(lift_b),
            'SUPPORT_CI_LOW': np.quantile(support_b, 0.025),
            'SUPPORT_CI_HIGH': np.quantile(support_b, 0.975),
            'CONFIDENCE_CI_LOW': np.quantile(confidence_b, 0.025),
            'CONFIDENCE_CI_HIGH': np.quantile(confidence_b, 0.975),
            'LIFT_CI_LOW': np.quantile(lift_b, 0.025),
            'LIFT_CI_HIGH': np.quantile(lift_b, 0.975),
            'BOOTSTRAP_LIFT_GT_1_RATE': float(np.mean(lift_b > 1)),
            'BOOTSTRAP_FULL_THRESHOLD_RATE': float(np.mean(
                (support_b >= MIN_SUPPORT)
                & (confidence_b >= MIN_CONFIDENCE)
                & (lift_b >= MIN_LIFT)
            )),
        })

    bootstrap_df = pd.DataFrame(bootstrap_rows)
    deep_rules = shortlisted_rules.merge(bootstrap_df, on='RULE_ID', how='left', validate='one_to_one')
    deep_rules['PASS_BOOTSTRAP'] = (
        (deep_rules['LIFT_CI_LOW'] > 1)
        & (deep_rules['BOOTSTRAP_LIFT_GT_1_RATE'] >= BOOTSTRAP_REQUIRED_LIFT_GT_1_RATE)
        & (deep_rules['BOOTSTRAP_FULL_THRESHOLD_RATE'] >= BOOTSTRAP_REQUIRED_FULL_THRESHOLD_RATE)
    )

    bootstrap_df.to_csv(RUN_DIR / '09_khoang_tin_cay_bootstrap.csv', index=False, encoding='utf-8-sig')
    deep_rules.to_csv(RUN_DIR / '10_luat_sau_bootstrap.csv', index=False, encoding='utf-8-sig')

    print('Rules passing bootstrap:', int(deep_rules['PASS_BOOTSTRAP'].sum()), '/', len(deep_rules))
    display(deep_rules.sort_values('LIFT_CI_LOW', ascending=False).head(10))


### Encoding sensitivity


In [ ]:
if HAS_RULES:
    def evaluate_rules_on_matrix(rules, frame, configuration):
        available = set(frame.columns)
        masks = {col: frame[col].to_numpy(dtype=bool) for col in frame.columns if col != 'RESPONDENT_ID'}
        rows = []
        for row in rules.itertuples():
            ants = [row.ANTECEDENT_1] + ([row.ANTECEDENT_2] if row.ANTECEDENT_SIZE == 2 else [])
            required = ants + [row.CONSEQUENT]
            if not set(required).issubset(available):
                rows.append({'RULE_ID': row.RULE_ID, 'CONFIGURATION': configuration, 'EVALUATION_STATUS': 'KHONG_AP_DUNG'})
                continue
            x = np.logical_and.reduce([masks[token] for token in ants])
            y = masks[row.CONSEQUENT]
            metrics = rule_metrics(x, y)
            metrics.update({
                'RULE_ID': row.RULE_ID,
                'CONFIGURATION': configuration,
                'EVALUATION_STATUS': 'DA_TINH',
            })
            rows.append(metrics)
        result = pd.DataFrame(rows)
        result['PASS_SENSITIVITY'] = False
        valid = result['EVALUATION_STATUS'].eq('DA_TINH')
        result.loc[valid, 'PASS_SENSITIVITY'] = (
            (result.loc[valid, 'JOINT_COUNT'] >= SENSITIVITY_MIN_COUNT)
            & (result.loc[valid, 'CONFIDENCE'] >= SENSITIVITY_MIN_CONFIDENCE)
            & (result.loc[valid, 'LIFT'] > SENSITIVITY_MIN_LIFT)
            & (result.loc[valid, 'FISHER_P'] <= FDR_ALPHA)
        )
        return result

    sens_no_enp = evaluate_rules_on_matrix(deep_rules, no_enp_matrix, C_NO_ENP)
    sens_strict = evaluate_rules_on_matrix(deep_rules, strict_matrix, C_STRICT)
    sensitivity_long = pd.concat([sens_no_enp, sens_strict], ignore_index=True)
    sensitivity_long.to_csv(RUN_DIR / '11_kiem_tra_do_nhay_dang_dai.csv', index=False, encoding='utf-8-sig')

    sens_wide = sensitivity_long.pivot(index='RULE_ID', columns='CONFIGURATION', values='PASS_SENSITIVITY')
    status_wide = sensitivity_long.pivot(index='RULE_ID', columns='CONFIGURATION', values='EVALUATION_STATUS')
    robustness = deep_rules.copy()
    robustness['PASS_NO_ENP'] = robustness['RULE_ID'].map(sens_wide.get(C_NO_ENP, pd.Series(dtype=object)))
    robustness['PASS_STRICT'] = robustness['RULE_ID'].map(sens_wide.get(C_STRICT, pd.Series(dtype=object)))
    robustness['NO_ENP_STATUS'] = robustness['RULE_ID'].map(status_wide.get(C_NO_ENP, pd.Series(dtype=object)))

    def classify_robustness(row):
        if not row['PASS_BOOTSTRAP']:
            return 'KHONG_VUOT_BOOTSTRAP'
        strict_ok = bool(row['PASS_STRICT'])
        no_enp_not_applicable = row['NO_ENP_STATUS'] == 'KHONG_AP_DUNG'
        no_enp_ok = bool(row['PASS_NO_ENP']) if not pd.isna(row['PASS_NO_ENP']) else False
        if strict_ok and (no_enp_ok or no_enp_not_applicable):
            return 'BEN_VUNG_CAO'
        if strict_ok or no_enp_ok:
            return 'BEN_VUNG_MOT_PHAN'
        return 'CHI_ON_DINH_O_CAU_HINH_CHINH'

    robustness['ROBUSTNESS_LEVEL'] = robustness.apply(classify_robustness, axis=1)
    robustness.to_csv(RUN_DIR / '12_tong_hop_do_ben_luat.csv', index=False, encoding='utf-8-sig')

    display(robustness['ROBUSTNESS_LEVEL'].value_counts(dropna=False).rename_axis('MUC_DO').reset_index(name='N_RULES'))


### Phase 3 profile comparison


In [ ]:
if HAS_RULES:
    profile_long_rows = []
    heterogeneity_rows = []

    if cluster_labels is not None:
        profile_map = cluster_labels[['RESPONDENT_ID', 'PROFILE_CODE']].copy()
        profiled = main_matrix.merge(profile_map, on='RESPONDENT_ID', how='left', validate='one_to_one')
        profile_codes = sorted(profiled['PROFILE_CODE'].dropna().astype(str).unique())
        if len(profile_codes) != 2:
            print(f'Warning: {len(profile_codes)} profiles found. Pairwise comparison requires exactly two profiles.')

        for row in robustness.itertuples():
            ants = [row.ANTECEDENT_1] + ([row.ANTECEDENT_2] if row.ANTECEDENT_SIZE == 2 else [])
            per_profile = {}
            for profile in profile_codes:
                block = profiled.loc[profiled['PROFILE_CODE'].astype(str) == profile]
                x = np.logical_and.reduce([block[token].to_numpy(dtype=bool) for token in ants])
                y = block[row.CONSEQUENT].to_numpy(dtype=bool)
                metrics = rule_metrics(x, y)
                per_profile[profile] = metrics
                profile_long_rows.append({'RULE_ID': row.RULE_ID, 'PROFILE_CODE': profile, **metrics})

            if len(profile_codes) == 2:
                first, second = profile_codes
                m1, m2 = per_profile[first], per_profile[second]
                sufficient = (
                    m1['ANTECEDENT_COUNT'] >= PROFILE_MIN_ANTECEDENT_COUNT
                    and m2['ANTECEDENT_COUNT'] >= PROFILE_MIN_ANTECEDENT_COUNT
                )
                if sufficient:
                    table = [
                        [m1['JOINT_COUNT'], m1['ANTECEDENT_COUNT'] - m1['JOINT_COUNT']],
                        [m2['JOINT_COUNT'], m2['ANTECEDENT_COUNT'] - m2['JOINT_COUNT']],
                    ]
                    p_heterogeneity = fisher_exact(table, alternative='two-sided').pvalue
                else:
                    p_heterogeneity = np.nan
                heterogeneity_rows.append({
                    'RULE_ID': row.RULE_ID,
                    'PROFILE_1': first,
                    'PROFILE_2': second,
                    'ANTECEDENT_COUNT_1': m1['ANTECEDENT_COUNT'],
                    'ANTECEDENT_COUNT_2': m2['ANTECEDENT_COUNT'],
                    'CONFIDENCE_1': m1['CONFIDENCE'],
                    'CONFIDENCE_2': m2['CONFIDENCE'],
                    'CONFIDENCE_DIFF_1_MINUS_2': m1['CONFIDENCE'] - m2['CONFIDENCE'],
                    'SUFFICIENT_PROFILE_COUNTS': sufficient,
                    'HETEROGENEITY_P': p_heterogeneity,
                })

    profile_long_df = pd.DataFrame(profile_long_rows) if profile_long_rows else pd.DataFrame(columns=['RULE_ID','PROFILE_CODE','N','ANTECEDENT_COUNT','CONSEQUENT_COUNT','JOINT_COUNT','ANTECEDENT_SUPPORT','CONSEQUENT_BASELINE','SUPPORT','CONFIDENCE','LIFT','LEVERAGE','CONVICTION','FISHER_P'])
    heterogeneity_df = pd.DataFrame(heterogeneity_rows) if heterogeneity_rows else pd.DataFrame(columns=['RULE_ID','PROFILE_1','PROFILE_2','ANTECEDENT_COUNT_1','ANTECEDENT_COUNT_2','CONFIDENCE_1','CONFIDENCE_2','CONFIDENCE_DIFF_1_MINUS_2','SUFFICIENT_PROFILE_COUNTS','HETEROGENEITY_P','HETEROGENEITY_Q','PROFILE_DIFFERENCE_FLAG'])
    if not heterogeneity_df.empty:
        valid = heterogeneity_df['HETEROGENEITY_P'].notna()
        heterogeneity_df['HETEROGENEITY_Q'] = np.nan
        heterogeneity_df.loc[valid, 'HETEROGENEITY_Q'] = bh_adjust(
            heterogeneity_df.loc[valid, 'HETEROGENEITY_P'].to_numpy()
        )
        heterogeneity_df['PROFILE_DIFFERENCE_FLAG'] = (
            heterogeneity_df['SUFFICIENT_PROFILE_COUNTS']
            & (heterogeneity_df['CONFIDENCE_DIFF_1_MINUS_2'].abs() >= PROFILE_MIN_CONFIDENCE_DIFFERENCE)
            & (heterogeneity_df['HETEROGENEITY_Q'] <= FDR_ALPHA)
        )

    profile_long_df.to_csv(RUN_DIR / '13_chi_so_luat_theo_ho_so.csv', index=False, encoding='utf-8-sig')
    heterogeneity_df.to_csv(RUN_DIR / '14_kiem_dinh_khac_biet_ho_so.csv', index=False, encoding='utf-8-sig')

    if not heterogeneity_df.empty:
        print('Rules showing evidence of a profile difference:', int(heterogeneity_df['PROFILE_DIFFERENCE_FLAG'].sum()))
        profile_display = heterogeneity_df.loc[heterogeneity_df['SUFFICIENT_PROFILE_COUNTS']].copy()
        display(profile_display.sort_values('CONFIDENCE_DIFF_1_MINUS_2', key=lambda x: x.abs(), ascending=False).head(10))
    else:
        print('Profile labels were unavailable or did not permit the two-profile comparison.')


### Rule descriptions and priorities


In [ ]:
if HAS_RULES:
    def short_question(item):
        value = str(dictionary_by_item[item].get('SOURCE_QUESTION', '')).strip()
        bracketed = re.findall(r'\[([^\[\]]+)\]', value)
        if bracketed:
            text = bracketed[-1]
        elif f'[{item}.' in value:
            text = value.split(f'[{item}.', 1)[1].strip()
            text = f'{text} [source wording was incomplete]' if not value.rstrip().endswith(']') else text
        else:
            text = value
        text = re.sub(r'^' + re.escape(item) + r'\.\s*', '', text).strip()
        return text.rstrip('.')

    def token_state_text(token):
        info = token_info[token]
        if info['scale'] == 'PTED':
            state = 'high difficulty' if info['state'] == 'HIGH' else 'low difficulty'
        elif info['scale'] == 'ENP':
            state = 'high need' if info['state'] == 'HIGH' else 'low need'
        elif info['role'] == 'CONSEQUENT':
            state = 'high feature need' if info['state'] == 'HIGH' else 'low feature need'
        else:
            state = 'high frequency' if info['state'] == 'HIGH' else 'low frequency'
        return f"{info['item']}: {short_question(info['item'])} [{state}]"

    def rule_text(row):
        ants = [row['ANTECEDENT_1']] + ([row['ANTECEDENT_2']] if int(row['ANTECEDENT_SIZE']) == 2 else [])
        left = ' AND '.join(token_state_text(token) for token in ants)
        right = token_state_text(row['CONSEQUENT'])
        return f'IF {left} THEN {right}'

    eligible_priority = robustness.loc[
        robustness['PASS_BOOTSTRAP']
        & robustness['ROBUSTNESS_LEVEL'].isin(['BEN_VUNG_CAO', 'BEN_VUNG_MOT_PHAN'])
    ].copy()
    eligible_priority['ROBUSTNESS_RANK'] = eligible_priority['ROBUSTNESS_LEVEL'].map({
        'BEN_VUNG_CAO': 0,
        'BEN_VUNG_MOT_PHAN': 1,
    }).fillna(2)
    eligible_priority.sort_values(
        ['CONSEQUENT', 'ROBUSTNESS_RANK', 'LIFT_CI_LOW', 'SUPPORT'],
        ascending=[True, True, False, False],
        inplace=True,
    )
    priority_rules = eligible_priority.groupby('CONSEQUENT', group_keys=False).head(2).copy()
    priority_rules.sort_values(['LIFT_CI_LOW', 'SUPPORT'], ascending=False, inplace=True)

    if not priority_rules.empty:
        priority_rules['RULE_TEXT'] = priority_rules.apply(rule_text, axis=1)
        priority_rules['INTERPRETATION'] = priority_rules.apply(
            lambda row: (
                f"{int(row['JOINT_COUNT'])}/{int(row['N'])} respondents have both antecedent and consequent. "
                f"Among respondents with the antecedent, high feature need occurs in {row['CONFIDENCE']:.1%}; "
                f"the baseline rate is {row['CONSEQUENT_BASELINE']:.1%}; lift is {row['LIFT']:.2f}."
            ), axis=1
        )

    priority_rules.to_csv(RUN_DIR / '15_luat_uu_tien.csv', index=False, encoding='utf-8-sig')

    high_robust_rules = robustness.loc[
        robustness['PASS_BOOTSTRAP']
        & robustness['ROBUSTNESS_LEVEL'].eq('BEN_VUNG_CAO')
    ].copy()
    if not high_robust_rules.empty:
        high_robust_rules['RULE_TEXT'] = high_robust_rules.apply(rule_text, axis=1)
        high_robust_rules['INTERPRETATION'] = high_robust_rules.apply(
            lambda row: (
                f"{int(row['JOINT_COUNT'])}/{int(row['N'])} respondents have both antecedent and consequent. "
                f"Confidence is {row['CONFIDENCE']:.1%}, the baseline rate is {row['CONSEQUENT_BASELINE']:.1%}, "
                f"lift is {row['LIFT']:.2f}."
            ), axis=1
        )
        high_robust_rules.sort_values(['LIFT_CI_LOW', 'SUPPORT'], ascending=False, inplace=True)
    high_robust_rules.to_csv(RUN_DIR / '15a_luat_ben_vung_cao.csv', index=False, encoding='utf-8-sig')

    feature_summary = pd.DataFrame(columns=['CONSEQUENT','N_PRIORITY_RULES','MAX_LIFT','MIN_LIFT_CI_LOW','MAX_CONFIDENCE','MAX_SUPPORT','CONSEQUENT_BASELINE','ITEM','FUNCTION_DESCRIPTION'])
    if not priority_rules.empty:
        feature_summary = priority_rules.groupby('CONSEQUENT').agg(
            N_PRIORITY_RULES=('RULE_ID', 'size'),
            MAX_LIFT=('LIFT', 'max'),
            MIN_LIFT_CI_LOW=('LIFT_CI_LOW', 'min'),
            MAX_CONFIDENCE=('CONFIDENCE', 'max'),
            MAX_SUPPORT=('SUPPORT', 'max'),
            CONSEQUENT_BASELINE=('CONSEQUENT_BASELINE', 'first'),
        ).reset_index()
        feature_summary['ITEM'] = feature_summary['CONSEQUENT'].str.split('__').str[0]
        feature_summary['FUNCTION_DESCRIPTION'] = feature_summary['ITEM'].map(short_question)
        feature_summary.sort_values(['N_PRIORITY_RULES', 'MAX_LIFT'], ascending=False, inplace=True)
    feature_summary.to_csv(RUN_DIR / '16_tom_tat_chuc_nang_tu_luat.csv', index=False, encoding='utf-8-sig')

    print('Priority rules:', len(priority_rules))
    display(priority_rules[['RULE_ID', 'RULE_TEXT', 'SUPPORT', 'CONFIDENCE', 'LIFT', 'LIFT_CI_LOW', 'ROBUSTNESS_LEVEL']].head(20))


### Figures


In [ ]:
if HAS_RULES:
    # Figure 1: screening flow
    fig, ax = plt.subplots(figsize=(10, 5.5))
    counts = screen_summary['N_RULES'].tolist() + [int(deep_rules['PASS_BOOTSTRAP'].sum()), len(priority_rules)]
    labels = [
        'Candidates', 'Passes effect thresholds', 'Also passes FDR',
        'Remove redundant rules', 'Detailed validation', 'Passes bootstrap', 'Priority'
    ]
    ax.plot(range(len(counts)), counts, color='black', marker='o', linewidth=2)
    ax.fill_between(range(len(counts)), counts, color='0.85')
    ax.set_xticks(range(len(labels)), labels, rotation=35, ha='right')
    ax.set_ylabel('Number of rules')
    ax.set_title('Association rule screening flow')
    for i, value in enumerate(counts):
        ax.annotate(f'{value:,}', (i, value), xytext=(0, 7), textcoords='offset points', ha='center', fontsize=9)
    fig.tight_layout()
    fig.savefig(RUN_DIR / '17_luong_sang_loc_luat.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    # Figure 2: support, confidence, and lift
    fig, ax = plt.subplots(figsize=(9.5, 6.5))
    plot_data = candidate_rules.copy()
    ax.scatter(plot_data['SUPPORT'], plot_data['CONFIDENCE'], s=8, color='0.75', alpha=0.35, label='Candidates')
    ax.scatter(shortlisted_rules['SUPPORT'], shortlisted_rules['CONFIDENCE'],
               s=24 + 35 * (shortlisted_rules['LIFT'] - 1).clip(lower=0),
               color='0.2', alpha=0.8, label='Selected for detailed validation')
    ax.axhline(MIN_CONFIDENCE, color='black', linestyle='--', linewidth=1)
    ax.axvline(MIN_SUPPORT, color='black', linestyle=':', linewidth=1)
    ax.set_xlabel('Support')
    ax.set_ylabel('Confidence')
    ax.set_title('Rule quality space')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(RUN_DIR / '18_support_confidence_lift.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    # Figure 3: highest-priority rules
    top_plot = priority_rules.sort_values('LIFT_CI_LOW', ascending=True).tail(20).copy()
    fig, ax = plt.subplots(figsize=(11, max(5, 0.38 * len(top_plot) + 1.5)))
    if not top_plot.empty:
        labels_plot = top_plot.apply(
            lambda r: f"{r['ANTECEDENT_1'].split('__')[0]}" +
                      (f" + {r['ANTECEDENT_2'].split('__')[0]}" if int(r['ANTECEDENT_SIZE']) == 2 else '') +
                      f" → {r['CONSEQUENT'].split('__')[0]}", axis=1
        )
        y = np.arange(len(top_plot))
        ax.hlines(y, top_plot['LIFT_CI_LOW'], top_plot['LIFT_CI_HIGH'], color='0.55', linewidth=2)
        ax.scatter(top_plot['LIFT'], y, color='black', s=35)
        ax.axvline(1, color='black', linestyle='--', linewidth=1)
        ax.set_yticks(y, labels_plot)
        ax.set_xlabel('Lift and 95% bootstrap interval')
    ax.set_title('Priority rules by lift stability')
    fig.tight_layout()
    fig.savefig(RUN_DIR / '19_luat_uu_tien_khoang_tin_cay.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    # Figure 4: single-antecedent rule heatmap
    single_screen = screened_rules.loc[screened_rules['ANTECEDENT_SIZE'] == 1].copy()
    top_ants = single_screen.groupby('ANTECEDENT_1')['LIFT'].max().nlargest(18).index
    heat = single_screen.loc[single_screen['ANTECEDENT_1'].isin(top_ants)].pivot_table(
        index='ANTECEDENT_1', columns='CONSEQUENT', values='LIFT', aggfunc='max'
    )
    fig, ax = plt.subplots(figsize=(14, 8))
    if not heat.empty:
        sns.heatmap(heat, cmap='Greys', vmin=1, linewidths=0.25, linecolor='white', ax=ax, cbar_kws={'label': 'Lift'})
        ax.set_xlabel('High feature need in tier III')
        ax.set_ylabel('Antecedent state in tier I or II')
    ax.set_title('Lift of notable single-antecedent rules')
    fig.tight_layout()
    fig.savefig(RUN_DIR / '20_ban_do_lift_luat_don.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    # Figure 5: rule robustness
    robust_counts = robustness['ROBUSTNESS_LEVEL'].value_counts().sort_values()
    robust_labels = {
        'BEN_VUNG_CAO': 'Highly robust',
        'BEN_VUNG_MOT_PHAN': 'Partly robust',
        'KHONG_VUOT_BOOTSTRAP': 'Does not pass bootstrap',
        'CHI_ON_DINH_O_CAU_HINH_CHINH': 'Stable only under primary encoding',
    }
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.barh([robust_labels.get(x, x) for x in robust_counts.index], robust_counts.values, color='0.65', edgecolor='black')
    ax.set_xlabel('Number of rules')
    ax.set_title('Robustness of validated rules')
    for i, value in enumerate(robust_counts.values):
        ax.text(value, i, f' {value}', va='center')
    fig.tight_layout()
    fig.savefig(RUN_DIR / '21_do_ben_luat.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    # Figure 6: confidence difference between profiles
    fig, ax = plt.subplots(figsize=(10, 6))
    if not heterogeneity_df.empty:
        profile_plot = heterogeneity_df.loc[heterogeneity_df['SUFFICIENT_PROFILE_COUNTS']].copy()
        profile_plot['ABS_DIFF'] = profile_plot['CONFIDENCE_DIFF_1_MINUS_2'].abs()
        profile_plot = profile_plot.nlargest(20, 'ABS_DIFF').sort_values('CONFIDENCE_DIFF_1_MINUS_2')
        ax.barh(profile_plot['RULE_ID'], profile_plot['CONFIDENCE_DIFF_1_MINUS_2'], color='0.65', edgecolor='black')
        ax.axvline(0, color='black', linewidth=1)
        ax.axvline(PROFILE_MIN_CONFIDENCE_DIFFERENCE, color='black', linestyle='--', linewidth=1)
        ax.axvline(-PROFILE_MIN_CONFIDENCE_DIFFERENCE, color='black', linestyle='--', linewidth=1)
        ax.set_xlabel('Confidence difference: higher-strategy minus lower-strategy profile' if len(profile_plot) else 'Confidence difference')
    else:
        ax.text(0.5, 0.5, 'No profile data available for comparison', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Largest rule differences between profiles')
    fig.tight_layout()
    fig.savefig(RUN_DIR / '22_khac_biet_luat_theo_ho_so.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

    print('Six figures were saved.')


### Integrity checks


In [ ]:
if HAS_RULES:
    run_config = {
        'run_timestamp_utc': RUN_STAMP,
        'input_dir': str(INPUT_DIR), 'input_csv_sha256': input_hash_before,
        'cluster_path': str(PHASE3_RESULTS_PATH), 'cluster_source': cluster_source,
        'main_input_sha256': sha256_file(INPUT_DIR / FILES['main']),
        'no_enp_input_sha256': sha256_file(INPUT_DIR / FILES['no_enp']),
        'strict_input_sha256': sha256_file(INPUT_DIR / FILES['strict']),
        'dictionary_input_sha256': sha256_file(INPUT_DIR / FILES['dictionary']),
        'min_support': MIN_SUPPORT,
        'min_confidence': MIN_CONFIDENCE,
        'min_lift': MIN_LIFT,
        'fdr_alpha': FDR_ALPHA,
        'max_antecedent_size': MAX_ANTECEDENT_SIZE,
        'pair_require_different_scales': PAIR_REQUIRE_DIFFERENT_SCALES,
        'min_confidence_gain_pair': MIN_CONFIDENCE_GAIN_PAIR,
        'n_bootstrap': N_BOOTSTRAP,
        'random_seed': RANDOM_SEED,
        'bootstrap_required_lift_gt_1_rate': BOOTSTRAP_REQUIRED_LIFT_GT_1_RATE,
        'bootstrap_required_full_threshold_rate': BOOTSTRAP_REQUIRED_FULL_THRESHOLD_RATE,
        'profile_labels_provided': cluster_labels is not None,
        'python_version': sys.version,
        'platform': platform.platform(),
        'pandas_version': pd.__version__,
        'numpy_version': np.__version__,
    }
    with open(RUN_DIR / '23_cau_hinh_lan_chay.json', 'w', encoding='utf-8') as handle:
        json.dump(run_config, handle, ensure_ascii=False, indent=2)

    guide = f"""# How to read phase 4B results

    ## Scope of inference

    This notebook tests conditional co-occurrence between tiers I or II and high feature need in tier III. The rules do not establish causality.

    ## Main results

    - Candidate rules meeting support: {len(candidate_rules)}.
    - Rules passing initial screening: {len(screened_rules)}.
    - Rules selected for detailed validation: {len(deep_rules)}.
    - Rules passing bootstrap: {int(deep_rules['PASS_BOOTSTRAP'].sum())}.
    - Final priority rules: {len(priority_rules)}.

    ## Recommended reading order

    1. `15a_luat_ben_vung_cao.csv`: rules passing bootstrap and both applicable sensitivity checks.
    2. `15_luat_uu_tien.csv`: at most two rules per feature with their robustness category.
    3. `16_tom_tat_chuc_nang_tu_luat.csv`: a summary by foundational feature.
    4. `12_tong_hop_do_ben_luat.csv`: bootstrap and sensitivity results.
    5. `14_kiem_dinh_khac_biet_ho_so.csv`: differences between the two profiles.

    Read confidence with the baseline prevalence and lift. High confidence with lift near one gives little improvement over the overall sample.
    """
    with open(RUN_DIR / '24_huong_dan_doc_ket_qua.md', 'w', encoding='utf-8') as handle:
        handle.write(guide)

    expected_outputs = [
        '01_kiem_tra_dau_vao.csv', '02_loi_toan_ven_4A.csv', '03_loi_cau_truc.csv',
        '04_tat_ca_luat_ung_vien.csv', '05_luat_ung_vien_kem_co_sang_loc.csv',
        '06_luat_dat_sang_loc_ban_dau.csv', '07_luat_duoc_chon_kiem_dinh_sau.csv',
        '08_tom_tat_sang_loc.csv', '09_khoang_tin_cay_bootstrap.csv',
        '10_luat_sau_bootstrap.csv', '11_kiem_tra_do_nhay_dang_dai.csv',
        '12_tong_hop_do_ben_luat.csv', '13_chi_so_luat_theo_ho_so.csv',
        '14_kiem_dinh_khac_biet_ho_so.csv', '15_luat_uu_tien.csv', '15a_luat_ben_vung_cao.csv',
        '16_tom_tat_chuc_nang_tu_luat.csv', '17_luong_sang_loc_luat.png',
        '18_support_confidence_lift.png', '19_luat_uu_tien_khoang_tin_cay.png',
        '20_ban_do_lift_luat_don.png', '21_do_ben_luat.png',
        '22_khac_biet_luat_theo_ho_so.png', '23_cau_hinh_lan_chay.json',
        '24_huong_dan_doc_ket_qua.md',
    ]

    readback_rows = []
    for filename in expected_outputs:
        path = RUN_DIR / filename
        status = 'DAT'
        detail = ''
        try:
            if not path.is_file() or path.stat().st_size == 0:
                raise ValueError('The file is missing or empty')
            if path.suffix.lower() == '.csv':
                pd.read_csv(path, encoding='utf-8-sig')
            elif path.suffix.lower() == '.json':
                json.loads(path.read_text(encoding='utf-8'))
            elif path.suffix.lower() == '.png':
                with open(path, 'rb') as handle:
                    if handle.read(8) != b'\x89PNG\r\n\x1a\n':
                        raise ValueError('Invalid PNG signature')
        except Exception as exc:
            status = 'KHONG_DAT'
            detail = str(exc)
        readback_rows.append({'FILE': filename, 'STATUS': status, 'DETAIL': detail})

    readback_df = pd.DataFrame(readback_rows)
    readback_df.to_csv(RUN_DIR / '25_kiem_tra_doc_lai_dau_ra.csv', index=False, encoding='utf-8-sig')
    if (readback_df['STATUS'] != 'DAT').any():
        raise RuntimeError('An output could not be read back; inspect 25_kiem_tra_doc_lai_dau_ra.csv.')

    manifest_rows = []
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file() and path.name != '26_manifest_sha256.csv':
            manifest_rows.append({'FILE': path.name, 'SIZE_BYTES': path.stat().st_size, 'SHA256': sha256_file(path)})
    manifest_out = pd.DataFrame(manifest_rows)
    manifest_out.to_csv(RUN_DIR / '26_manifest_sha256.csv', index=False, encoding='utf-8-sig')

    final_summary = pd.DataFrame([
        {'NOI_DUNG': 'Primary sample size', 'KET_QUA': len(main_matrix)},
        {'NOI_DUNG': 'Candidate rules', 'KET_QUA': len(candidate_rules)},
        {'NOI_DUNG': 'Rules passing initial screening', 'KET_QUA': len(screened_rules)},
        {'NOI_DUNG': 'Rules entering detailed validation', 'KET_QUA': len(deep_rules)},
        {'NOI_DUNG': 'Rules passing bootstrap', 'KET_QUA': int(deep_rules['PASS_BOOTSTRAP'].sum())},
        {'NOI_DUNG': 'Highly robust rules', 'KET_QUA': len(high_robust_rules)},
        {'NOI_DUNG': 'Priority rules', 'KET_QUA': len(priority_rules)},
        {'NOI_DUNG': 'Files verified by readback', 'KET_QUA': int((readback_df['STATUS'] == 'DAT').sum())},
    ])
    display(final_summary)
    print('PHASE 4B COMPLETE')
    print('Results directory:', RUN_DIR)


### Empty-rule outcome


In [ ]:
if not HAS_RULES:
    empty_files=['09_khoang_tin_cay_bootstrap.csv','10_luat_sau_bootstrap.csv',
      '11_kiem_tra_do_nhay_dang_dai.csv','12_tong_hop_do_ben_luat.csv',
      '13_chi_so_luat_theo_ho_so.csv','14_kiem_dinh_khac_biet_ho_so.csv',
      '15_luat_uu_tien.csv','15a_luat_ben_vung_cao.csv','16_tom_tat_chuc_nang_tu_luat.csv']
    for name in empty_files:
        pd.DataFrame(columns=['RULE_ID','STATUS']).to_csv(RUN_DIR/name,index=False,encoding='utf-8-sig')
    for name in ['17_luong_sang_loc_luat.png','18_support_confidence_lift.png',
      '19_luat_uu_tien_khoang_tin_cay.png','20_ban_do_lift_luat_don.png',
      '21_do_ben_luat.png','22_khac_biet_luat_theo_ho_so.png']:
        fig,ax=plt.subplots(figsize=(8,2))
        ax.text(.5,.5,'No rule passed the configured screening',ha='center',va='center')
        ax.axis('off');fig.savefig(RUN_DIR/name,dpi=FIGURE_DPI);plt.close(fig)
    priority_rules=pd.DataFrame(columns=['RULE_ID'])
    (RUN_DIR/'23_cau_hinh_lan_chay.json').write_text(json.dumps({'source_sha256':input_hash_before,
      'n_main':MAIN_N,'thresholds':{'support':MIN_SUPPORT,'confidence':MIN_CONFIDENCE,'lift':MIN_LIFT},
      'status':'NO_SHORTLISTED_RULES'},ensure_ascii=False,indent=2),encoding='utf-8')
    (RUN_DIR/'24_huong_dan_doc_ket_qua.md').write_text(
      '# Phase 4B\n\nNo rule passed the configured thresholds. Do not lower thresholds after observing results.\n',encoding='utf-8')
    pd.DataFrame([{'FILE':x.name,'STATUS':'DAT'} for x in RUN_DIR.iterdir() if x.is_file()]).to_csv(
      RUN_DIR/'25_kiem_tra_doc_lai_dau_ra.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame([{'FILE':x.name,'SIZE_BYTES':x.stat().st_size,'SHA256':sha256_file(x)}
      for x in sorted(RUN_DIR.iterdir()) if x.is_file() and x.name!='26_manifest_sha256.csv']).to_csv(
      RUN_DIR/'26_manifest_sha256.csv',index=False,encoding='utf-8-sig')
    print('No rule qualified; empty tables and a notice were exported.')


## Package the phase 4 results


In [ ]:
if sha256_file(input_path)!=input_hash_before:raise IOError('Input CSV changed after analysis.')
package=output_root/f'phase_04_{RUN_STAMP}.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in (DIR_4A,DIR_4B):
        for f in sorted(folder.iterdir()):
            if f.is_file():z.write(f,arcname=f'{folder.name}/{f.name}')
    z.writestr('00_tom_tat_lan_chay.json',json.dumps({'source_sha256':input_hash_before,
        'rows':TOTAL_N,'complete_enp':MAIN_N,'phase3':cluster_source,
        'candidate_rules':len(candidate_rules),'priority_rules':len(priority_rules),
        'bootstrap_replicates':N_BOOTSTRAP,'created_utc':RUN_STAMP},ensure_ascii=False,indent=2))
with zipfile.ZipFile(package) as z:
    if z.testzip():raise IOError('The phase 4 ZIP failed its integrity check.')
print('Phase 4 complete:',package)
